# Visible–Infrared Person Re-identification on SYSU-MM01

**COMP9444 Group Project**

A PMT baseline with planned smooth-transition and local-supervision extensions.

This notebook reports the baseline training run and a retrieval comparison of four baselines (V0, M0, AGW, CAJ) on the internal validation split. CAJ is reported twice: with the authors' code, which collapsed, and with our one-line correction. Extension experiments are in progress.

## 1. Introduction, Motivation and Problem Statement

Visible–infrared person re-identification matches an infrared query image to visible-light images of the same person across cameras. Differences in spectrum, lighting, pose and occlusion make appearance matching difficult. The objective is to learn features that preserve identity across modalities and rank matching gallery images highly, including for identities unseen during training.

We use the Progressive Modality-shared Transformer (PMT) [1] as our baseline. We investigate whether a smoother grey-to-colour transition and auxiliary local supervision improve retrieval, individually and together. Performance will be assessed using mAP and Rank-1/5/10 on identity-disjoint evaluation data.

## 2. Data Sources

We use **SYSU-MM01**, supplied through the course-linked [Kaggle mirror](https://www.kaggle.com/datasets/coconutjean/sysumm01), with the official identity splits [2]. Cameras 1, 2, 4 and 5 capture RGB images; cameras 3 and 6 capture infrared images.

The dataset contains 45,863 images: 30,071 RGB and 15,792 IR. Development uses 296 training identities and 99 validation identities; 96 test identities are reserved for final evaluation. The downloaded directories contain 22 additional identities absent from the official split lists. Their 1,118 images are excluded.

In [1]:
# Saved measurements from the baseline run; no training or data download.
import json
from IPython.display import Markdown, display

def show_table(headers, rows):
    lines = ["| " + " | ".join(headers) + " |", "| " + " | ".join(["---"] * len(headers)) + " |"]
    lines += ["| " + " | ".join(map(str, row)) + " |" for row in rows]
    display(Markdown("\n".join(lines)))

measurements = json.loads('{"images_per_split_modality": {"test/ir": 3803, "test/rgb": 6775, "train/ir": 9929, "train/rgb": 20284, "unassigned/ir": 80, "unassigned/rgb": 1038, "val/ir": 1980, "val/rgb": 1974}, "images_per_camera": {"cam1": 6289, "cam2": 7528, "cam3": 9812, "cam4": 8030, "cam5": 8224, "cam6": 5980}, "epochs": [{"epoch": 1, "stage": 1, "id": 9.895071681265206, "triplet": 2.4337873198974753, "msel": 0.0, "dcl": 0.0, "total": 12.328859010578897}, {"epoch": 2, "stage": 1, "id": 5.405836521543409, "triplet": 0.5574010249174319, "msel": 0.0, "dcl": 0.0, "total": 5.963237548891402}, {"epoch": 3, "stage": 1, "id": 2.313972295554705, "triplet": 0.27469258700014865, "msel": 0.0, "dcl": 0.0, "total": 2.588664880672714}, {"epoch": 4, "stage": 1, "id": 1.109292010654582, "triplet": 0.15504540034457984, "msel": 0.0, "dcl": 0.0, "total": 1.2643374105294545}, {"epoch": 5, "stage": 1, "id": 0.7351743248166255, "triplet": 0.11681329169771142, "msel": 0.0, "dcl": 0.0, "total": 0.8519876161693208}, {"epoch": 6, "stage": 1, "id": 0.5656185880138599, "triplet": 0.0967231511501051, "msel": 0.0, "dcl": 0.0, "total": 0.6623417392114139}, {"epoch": 7, "stage": 2, "id": 0.469481795202311, "triplet": 0.15250743363835328, "msel": 0.20957776596902106, "dcl": 0.6007366138046951, "total": 1.0271464172415259}, {"epoch": 8, "stage": 2, "id": 0.3523105321184248, "triplet": 0.09375102059639061, "msel": 0.11106696917335569, "dcl": 0.5779060389004989, "total": 0.7905480577294107}, {"epoch": 9, "stage": 2, "id": 0.27896022413508587, "triplet": 0.07092857236884192, "msel": 0.08879721369506059, "dcl": 0.5603820348708739, "total": 0.6744784196782828}, {"epoch": 10, "stage": 2, "id": 0.23269545188317836, "triplet": 0.057511916472010115, "msel": 0.07533332185546086, "dcl": 0.5446752944922937, "total": 0.600211675509834}, {"epoch": 11, "stage": 2, "id": 0.19635072999632378, "triplet": 0.04536981642795035, "msel": 0.06350961768643645, "dcl": 0.5258730547782181, "total": 0.536411883047595}, {"epoch": 12, "stage": 2, "id": 0.1853902283643989, "triplet": 0.04274636491144438, "msel": 0.057937336941418865, "dcl": 0.5124638109316382, "total": 0.5133371682716961}, {"epoch": 13, "stage": 2, "id": 0.15482122581643398, "triplet": 0.03413528978870961, "msel": 0.050652518109263966, "dcl": 0.4948721874099207, "total": 0.46171886679873836}, {"epoch": 14, "stage": 2, "id": 0.12480144161881075, "triplet": 0.029785835367147802, "msel": 0.047160174753546336, "dcl": 0.4771739614518333, "total": 0.4167543458995096}, {"epoch": 15, "stage": 2, "id": 0.1084839961635711, "triplet": 0.0266535037646282, "msel": 0.04264157512306131, "dcl": 0.4623540560781108, "total": 0.3876353150813312}, {"epoch": 16, "stage": 2, "id": 0.08627329345006866, "triplet": 0.022295507450115767, "msel": 0.03877250808269068, "dcl": 0.4441792790264503, "total": 0.35004469410973893}, {"epoch": 17, "stage": 2, "id": 0.087549884818306, "triplet": 0.022049351646285357, "msel": 0.036531552213345655, "dcl": 0.4318871846037077, "total": 0.34380860558429976}, {"epoch": 18, "stage": 2, "id": 0.07135801389519167, "triplet": 0.02020650813234661, "msel": 0.03365251646446297, "dcl": 0.4208294298694032, "total": 0.3188054945325776}, {"epoch": 19, "stage": 2, "id": 0.07076973426906101, "triplet": 0.01932831401815329, "msel": 0.032515983593077835, "dcl": 0.41057131720756845, "total": 0.3116416991416124}, {"epoch": 20, "stage": 2, "id": 0.05993384103942515, "triplet": 0.019468981734407833, "msel": 0.03045111172060125, "dcl": 0.40370109211212085, "total": 0.29647892459308933}, {"epoch": 21, "stage": 2, "id": 0.06196701410005832, "triplet": 0.017726471562653567, "msel": 0.02987099016500169, "dcl": 0.396795853054354, "total": 0.29302690770200457}, {"epoch": 22, "stage": 2, "id": 0.05798238050909845, "triplet": 0.01827493317367871, "msel": 0.029066515792202423, "dcl": 0.3944796729144327, "total": 0.28803040861706786}, {"epoch": 23, "stage": 2, "id": 0.05806540522233717, "triplet": 0.017107306881020606, "msel": 0.028732913109292917, "dcl": 0.39249476842217346, "total": 0.28578655275782516}, {"epoch": 24, "stage": 2, "id": 0.05570394186984224, "triplet": 0.018262740649433003, "msel": 0.02829119528233299, "dcl": 0.3902712081092605, "total": 0.28324788380384824}]}')

In [2]:
counts = measurements["images_per_split_modality"]
show_table(["Split", "Identities", "RGB images", "IR images"], [
    [split, ids, counts[f"{split}/rgb"], counts[f"{split}/ir"]]
    for split, ids in [("train",296), ("val",99), ("test",96), ("unassigned",22)]])

| Split | Identities | RGB images | IR images |
| --- | --- | --- | --- |
| train | 296 | 20284 | 9929 |
| val | 99 | 1974 | 1980 |
| test | 96 | 6775 | 3803 |
| unassigned | 22 | 1038 | 80 |

## 3. Exploratory Analysis of Data

### 3.1 Data balance and integrity

The training split has 20,284 RGB images and 9,929 IR images, approximately a 2:1 imbalance. We therefore sample equally from both modalities and balance identities within each training batch. All training identities have images in both modalities. The audit found no unreadable images, overlapping official identity splits or duplicate files shared across splits.

Camera-level image counts also vary, as shown below. These counts describe dataset coverage; they do not establish how difficult each camera is for retrieval.

In [3]:
show_table(["Camera", "Modality", "Images"], [
    [camera, "IR" if camera in ("cam3", "cam6") else "RGB", count]
    for camera, count in measurements["images_per_camera"].items()])

| Camera | Modality | Images |
| --- | --- | --- |
| cam1 | RGB | 6289 |
| cam2 | RGB | 7528 |
| cam3 | IR | 9812 |
| cam4 | RGB | 8030 |
| cam5 | RGB | 8224 |
| cam6 | IR | 5980 |

### 3.2 Further exploration

Per-identity sample distributions, image dimensions and intensity distributions, and representative RGB/IR and occlusion examples remain to be added with their implications for preprocessing and modelling.

## 4. Models and Methods

### 4.1 PMT (M0)

PMT is the main method of this project: both proposed extensions are built on it, and it is referred to as M0 in the experiments. We implement PMT following Lu et al. [1] and the pinned author PyTorch implementation. The shared ViT-B/16 has 12 blocks, 12 attention heads and 768-dimensional features. A 16×16 patch projection with stride 12 produces a 21×10 patch grid from 256×128 images. The CLS feature feeds a BN neck and an identity classifier. Metric losses use the pre-BN feature; retrieval uses the L2-normalised, post-BN global feature.

Training has two stages. Epochs 1–6 use greyscale RGB and IR inputs with identity classification and separate within-modality hard-triplet losses. Epochs 7–24 restore colour RGB and use joint hard-triplet, modality discrepancy (MSEL) and centre-based (DCL) losses:

$$L_1=L_{ID}^{R}+L_{ID}^{I}+L_{tri}^{R}+L_{tri}^{I}.$$
$$L_2=L_{ID}^{R}+L_{ID}^{I}+L_{tri}^{all}+0.5L_{MSEL}+0.5L_{DCL}.$$

MSEL aligns within- and cross-modality positive distances; DCL encourages identity-centred separation. We retain the stage-2 ID term as in the author's PyTorch code. PMT and its losses are prior work; our contribution here is the implementation and experiment. The augmentation recipe includes the author's random-erasing implementation. Source attribution and implementation differences are recorded in `ATTRIBUTION.md` and `docs/baseline.md`.

### 4.2 Comparison baselines

We compare M0 with three baselines trained on the same 296 identities, with the same identity-balanced sampler and evaluator, using seed 17.

- **V0** is M0 without PMT's components: the same ViT-B/16 trained for 24 epochs with identity and global hard-triplet losses only, without the greyscale stage, MSEL or DCL. It measures what those components add.
- **AGW** [3] is a two-stream ResNet-50 with non-local blocks and GeM pooling, trained with identity and weighted regularised triplet (WRT) losses. It is a widely used CNN baseline for this task.
- **CAJ** [4] extends AGW with channel augmentation (a second RGB view with one colour channel copied to all three; random channel copying or greyscale for IR) and a squared-difference triplet loss (ADP). It tests whether simple colour augmentation already gives the benefit that the smooth-transition extension targets.

AGW and CAJ were ported from the authors' code at a pinned commit [3] and keep the authors' training settings (Section 4.3). CAJ collapsed with that code; we also report a corrected CAJ that changes only the hard-negative weighting (Section 5.3). The authors trained on 395 identities; we use 296 so that validation identities are never seen in training.

### 4.3 Preprocessing and training settings

Each model keeps the preprocessing and optimisation settings of its original implementation; only the data split, sampler and evaluator are shared. All images are normalised using ImageNet statistics, and evaluation uses resize and normalisation only.

| Setting | V0 / M0 (ViT-B/16) | AGW / CAJ (ResNet-50) |
|---|---|---|
| Initialisation | Author-linked ImageNet ViT-B/16 weights | ImageNet ResNet-50 weights; last-stage stride 1 |
| Input size | 256×128 | 288×144 |
| Training augmentation | Horizontal flip, random erasing; IR also colour jitter or Gaussian blur | Pad and random crop, horizontal flip; CAJ adds channel augmentation and random erasing (p = 0.5) |
| Batch | 8 identities × 4 images × 2 modalities = 64 | Same sampler: 64 images; CAJ 96 (two RGB views) |
| Training length | 24 epochs; M0's first 6 in stage 1 | AGW 81 epochs; CAJ 100 epochs |
| Optimiser | AdamW; weight decay 0.0001 | SGD, Nesterov momentum 0.9; weight decay 0.0005 |
| Learning rate | Base 0.0003; half rate for patch projection and transformer blocks | Base 0.1; ×0.1 for the backbone |
| Schedule | 3-epoch warmup, cosine decay; minimum LR 0.000003 | 10-epoch linear warmup; ×0.1 at epochs 20 and 50 |
| Metric loss | Hard triplet, margin 0.1 | AGW: WRT; CAJ: ADP (both soft-margin, no fixed margin) |
| Precision | Mixed precision | Full precision |
| Training seed and device | 17; NVIDIA L40S | 17; NVIDIA L40S |

### 4.4 Proposed extensions

The smooth-transition extension gradually reduces the probability of greyscale RGB after stage 1. The local-supervision extension adds auxiliary identity losses on horizontal feature regions. The planned M0–M3 comparison varies these two factors while keeping the backbone and global retrieval feature fixed. These extensions have not yet been evaluated.

### 4.5 Retrieval protocol

Development evaluation uses IR→RGB all-search, single-shot retrieval on the internal validation split (99 identities). Ten fixed gallery draws (seeds 0–9) are selected from the RGB candidate pool following the author's sampling procedure, and every model is scored on the same ten galleries. Cam2 gallery images are excluded for cam3 queries, following the SYSU protocol, and CMC is computed at identity level. Features are taken after the BN neck, L2-normalised and ranked by cosine similarity, without flip augmentation or re-ranking. We report mAP, mINP and Rank-k, averaged over the ten galleries. The evaluator was checked against hand-computed cases and against the author's evaluation code.

In this split, IR queries come only from camera 3 and RGB candidates only from cameras 4 and 5. Validation scores are therefore not comparable with published SYSU-MM01 test results.

## 5. Results

### 5.1 Baseline training

The M0 baseline completed 24 epochs on the 296-identity training split using seed 17. The table shows mean training losses at the start and end of each stage; each row averages 633 batches. MSEL and DCL are shown before applying their 0.5 weights.

In [4]:
selected_epochs = {1, 6, 7, 24}
show_table(["Epoch", "Stage", "ID", "Triplet", "MSEL", "DCL", "Total"], [
    [row["epoch"], row["stage"], *[f"{row[key]:.4f}" for key in ["id","triplet","msel","dcl","total"]]]
    for row in measurements["epochs"] if row["epoch"] in selected_epochs])

| Epoch | Stage | ID | Triplet | MSEL | DCL | Total |
| --- | --- | --- | --- | --- | --- | --- |
| 1 | 1 | 9.8951 | 2.4338 | 0.0000 | 0.0000 | 12.3289 |
| 6 | 1 | 0.5656 | 0.0967 | 0.0000 | 0.0000 | 0.6623 |
| 7 | 2 | 0.4695 | 0.1525 | 0.2096 | 0.6007 | 1.0271 |
| 24 | 2 | 0.0557 | 0.0183 | 0.0283 | 0.3903 | 0.2832 |

Training loss decreases within both stages: from 12.3289 to 0.6623 in stage 1 and from 1.0271 to 0.2832 in stage 2. The objective and RGB input change at epoch 7, so the boundary values are not directly comparable. These measurements describe training fit; retrieval performance must be established separately.

### 5.2 Retrieval performance

All models were trained once (seed 17) on an NVIDIA L40S and scored on the same ten fixed galleries. A 2-epoch pilot of each new model was evaluated before its full run, to confirm that it learns before committing GPU time.


In [5]:
# Saved retrieval metrics (src/evaluate.py, metrics.json of each run); no evaluation is rerun here.
retrieval = json.loads('{"V0": {"run_id": "dev_V0_seed17_katana_20261030", "epochs": 24, "mAP": 0.33032, "mINP": 0.246556, "rank1": 0.290101, "rank5": 0.570505, "rank10": 0.698434, "rank20": 0.81803, "sd_mAP": 0.014614, "sd_rank1": 0.015728, "per_gallery_mAP": [0.335563, 0.356684, 0.304134, 0.334314, 0.335288, 0.334764, 0.321853, 0.314879, 0.32478, 0.340942], "per_gallery_rank1": [0.30404, 0.29899, 0.274747, 0.294949, 0.280808, 0.305051, 0.283838, 0.276263, 0.267172, 0.315152], "backbone": "ViT-B/16", "input": "256×128", "params_m": 86.04, "gflops": 37.73, "gpu_hours": 0.77}, "M0": {"run_id": "dev_M0_seed17_katana_20261025_retry1", "epochs": 24, "mAP": 0.343836, "mINP": 0.242371, "rank1": 0.320657, "rank5": 0.602727, "rank10": 0.726364, "rank20": 0.838788, "sd_mAP": 0.016459, "sd_rank1": 0.02384, "per_gallery_mAP": [0.339704, 0.370691, 0.310831, 0.346919, 0.357343, 0.340725, 0.333889, 0.332076, 0.352829, 0.353355], "per_gallery_rank1": [0.312626, 0.348485, 0.282323, 0.316667, 0.352525, 0.332323, 0.3, 0.297475, 0.317677, 0.346465], "backbone": "ViT-B/16", "input": "256×128", "params_m": 86.04, "gflops": 37.73, "gpu_hours": 0.99}, "AGW": {"run_id": "dev_AGW_seed17_katana_20261030", "epochs": 81, "mAP": 0.302351, "mINP": 0.213051, "rank1": 0.269394, "rank5": 0.54096, "rank10": 0.667273, "rank20": 0.792576, "sd_mAP": 0.016188, "sd_rank1": 0.022214, "per_gallery_mAP": [0.287833, 0.320856, 0.270877, 0.321848, 0.307348, 0.311798, 0.303074, 0.298666, 0.288355, 0.312854], "per_gallery_rank1": [0.24899, 0.288384, 0.229293, 0.29899, 0.271717, 0.285859, 0.269697, 0.262626, 0.248485, 0.289899], "backbone": "ResNet-50", "input": "288×144", "params_m": 24.16, "gflops": 10.27, "gpu_hours": 1.89}, "CAJ": {"run_id": "dev_CAJ_seed17_katana_20261030", "epochs": 100, "mAP": 0.109572, "mINP": 0.062451, "rank1": 0.070354, "rank5": 0.220606, "rank10": 0.339697, "rank20": 0.503131, "sd_mAP": 0.007005, "sd_rank1": 0.01183, "per_gallery_mAP": [0.11069, 0.111725, 0.1003, 0.124176, 0.105491, 0.115044, 0.106567, 0.109678, 0.10075, 0.111299], "per_gallery_rank1": [0.072222, 0.069192, 0.061111, 0.094444, 0.052525, 0.082828, 0.065152, 0.069192, 0.062121, 0.074747], "backbone": "ResNet-50", "input": "288×144", "params_m": 24.16, "gflops": 10.27, "gpu_hours": 3.57}, "CAJ_fix": {"run_id": "dev_CAJfix_seed17_katana_20261031", "epochs": 100, "mAP": 0.343824, "mINP": 0.236481, "rank1": 0.331414, "rank5": 0.593333, "rank10": 0.723081, "rank20": 0.840808, "sd_mAP": 0.018786, "sd_rank1": 0.024113, "per_gallery_mAP": [0.325437, 0.368329, 0.31226, 0.351055, 0.35108, 0.348719, 0.34039, 0.321479, 0.352877, 0.366615], "per_gallery_rank1": [0.312626, 0.358586, 0.291414, 0.347475, 0.343939, 0.344444, 0.337374, 0.29596, 0.325758, 0.356566], "backbone": "ResNet-50", "input": "288×144", "params_m": 24.16, "gflops": 10.27, "gpu_hours": 3.58}}')

pct = lambda value: f"{100 * value:.1f}"
labels = {"V0": "V0", "M0": "M0", "AGW": "AGW", "CAJ": "CAJ (authors' code)†", "CAJ_fix": "CAJ (corrected)‡"}
show_table(["Method", "Backbone", "Input", "Epochs", "mAP", "mINP", "Rank-1", "Rank-5", "Rank-10", "Rank-20", "Params (M)", "GFLOPs / image", "GPU hours"], [
    [labels[name], row["backbone"], row["input"], row["epochs"],
     f"{pct(row['mAP'])} ± {pct(row['sd_mAP'])}", pct(row["mINP"]), f"{pct(row['rank1'])} ± {pct(row['sd_rank1'])}",
     pct(row["rank5"]), pct(row["rank10"]), pct(row["rank20"]), row["params_m"], row["gflops"], row["gpu_hours"]]
    for name, row in retrieval.items()])

| Method | Backbone | Input | Epochs | mAP | mINP | Rank-1 | Rank-5 | Rank-10 | Rank-20 | Params (M) | GFLOPs / image | GPU hours |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| V0 | ViT-B/16 | 256×128 | 24 | 33.0 ± 1.5 | 24.7 | 29.0 ± 1.6 | 57.1 | 69.8 | 81.8 | 86.04 | 37.73 | 0.77 |
| M0 | ViT-B/16 | 256×128 | 24 | 34.4 ± 1.6 | 24.2 | 32.1 ± 2.4 | 60.3 | 72.6 | 83.9 | 86.04 | 37.73 | 0.99 |
| AGW | ResNet-50 | 288×144 | 81 | 30.2 ± 1.6 | 21.3 | 26.9 ± 2.2 | 54.1 | 66.7 | 79.3 | 24.16 | 10.27 | 1.89 |
| CAJ (authors' code)† | ResNet-50 | 288×144 | 100 | 11.0 ± 0.7 | 6.2 | 7.0 ± 1.2 | 22.1 | 34.0 | 50.3 | 24.16 | 10.27 | 3.57 |
| CAJ (corrected)‡ | ResNet-50 | 288×144 | 100 | 34.4 ± 1.9 | 23.6 | 33.1 ± 2.4 | 59.3 | 72.3 | 84.1 | 24.16 | 10.27 | 3.58 |

Values are percentages: mAP and Rank-1 show the mean ± standard deviation over the ten fixed galleries for one trained model. This spread reflects which gallery images were drawn, not variation between training runs. Parameters include the identity classifier. GFLOPs count one forward pass of one image at the model's input size (2 × multiply–accumulate). GPU hours are total L40S job wall time, including the pilot and feature export. † CAJ collapsed during training with the authors' code (Section 5.3), so its row is not a valid baseline result. ‡ Our correction of the hard-negative weighting; all other code and settings are the authors' (Section 5.3).

- **PMT components (M0 vs V0).** M0 is 1.4 mAP and 3.1 Rank-1 points above V0 and higher on all ten galleries; V0's mINP is slightly higher (24.7 vs 24.2). This supports a small benefit from the greyscale stage, MSEL and DCL, but it comes from a single training seed.
- **Transformer PMT vs CNN baseline (M0 vs AGW).** M0 is 4.1 mAP and 5.1 Rank-1 points above AGW, with about 3.6 times the parameters and computation. Backbone, input size, optimiser and losses all differ, so the gap cannot be attributed to the backbone alone.
- **Corrected CAJ vs M0.** The corrected CAJ matches M0: mAP is equal (34.4) and Rank-1 is 1.1 points higher (33.1 vs 32.1), with M0 ahead on 4 of 10 galleries for mAP and 3 of 10 for Rank-1. It does this with a ResNet-50 at about a quarter of M0's computation, but with 100 epochs and about 3.6 times M0's GPU time.
- **Corrected CAJ vs AGW.** The corrected CAJ is 4.1 mAP and 6.2 Rank-1 points above AGW on all ten galleries, which agrees with the published ordering. The gap combines channel augmentation, random erasing, the ADP loss, a larger batch, more epochs and the corrected weighting (AGW was trained with the original weighting), so it is not the effect of channel augmentation alone.
- **CAJ with the authors' code** is about 20 points below AGW because of the training collapse described below.

In [6]:
# Paired differences on the same ten galleries (percentage points).
import numpy as np
rows = []
for a, b in (("V0", "M0"), ("AGW", "M0"), ("CAJ_fix", "M0"), ("CAJ_fix", "AGW")):
    for metric, label in (("mAP", "mAP"), ("rank1", "Rank-1")):
        diff = 100 * (np.array(retrieval[a][f"per_gallery_{metric}"]) - np.array(retrieval[b][f"per_gallery_{metric}"]))
        rows.append([f"{labels[a]} − {b}", label, f"{round(diff.mean(), 1) + 0.0:+.1f}", f"{diff.min():+.1f} to {diff.max():+.1f}", f"{(diff > 0).sum()} / 10"])
show_table(["Comparison (A − B)", "Metric", "Mean", "Range over galleries", "Galleries where A is higher"], rows)

| Comparison (A − B) | Metric | Mean | Range over galleries | Galleries where A is higher |
| --- | --- | --- | --- | --- |
| V0 − M0 | mAP | -1.4 | -2.8 to -0.4 | 0 / 10 |
| V0 − M0 | Rank-1 | -3.1 | -7.2 to -0.8 | 0 / 10 |
| AGW − M0 | mAP | -4.1 | -6.4 to -2.5 | 0 / 10 |
| AGW − M0 | Rank-1 | -5.1 | -8.1 to -1.8 | 0 / 10 |
| CAJ (corrected)‡ − M0 | mAP | +0.0 | -1.4 to +1.3 | 6 / 10 |
| CAJ (corrected)‡ − M0 | Rank-1 | +1.1 | -0.9 to +3.7 | 7 / 10 |
| CAJ (corrected)‡ − AGW | mAP | +4.1 | +2.3 to +6.5 | 10 / 10 |
| CAJ (corrected)‡ − AGW | Rank-1 | +6.2 | +3.3 to +7.7 | 10 / 10 |

### 5.3 CAJ training collapse and correction

With the authors' code, CAJ did not train stably on our split. The table shows per-epoch means for CAJ with the authors' code and with the corrected weighting, with AGW's training identity accuracy for reference.

In [7]:
caj_curve = json.loads('[{"epoch": 1, "caj_id": 3.0587, "caj_adp": 88.0, "caj_acc": 0.3738, "agw_acc": 0.4867, "lr": 0.01, "fix_adp": 1.6498, "fix_acc": 0.0976}, {"epoch": 5, "caj_id": 0.3803, "caj_adp": 88.0, "caj_acc": 0.8992, "agw_acc": 0.9761, "lr": 0.05, "fix_adp": 0.4719, "fix_acc": 0.7852}, {"epoch": 6, "caj_id": 0.7982, "caj_adp": 81.4826, "caj_acc": 0.8459, "agw_acc": 0.9849, "lr": 0.06, "fix_adp": 0.3861, "fix_acc": 0.8434}, {"epoch": 7, "caj_id": 5.2523, "caj_adp": 0.702, "caj_acc": 0.0284, "agw_acc": 0.9927, "lr": 0.07, "fix_adp": 0.3148, "fix_acc": 0.875}, {"epoch": 10, "caj_id": 3.5379, "caj_adp": 0.6885, "caj_acc": 0.2135, "agw_acc": 0.9924, "lr": 0.1, "fix_adp": 0.2098, "fix_acc": 0.9086}, {"epoch": 20, "caj_id": 0.7562, "caj_adp": 0.4719, "caj_acc": 0.8031, "agw_acc": 0.879, "lr": 0.1, "fix_adp": 0.0633, "fix_acc": 0.9687}, {"epoch": 50, "caj_id": 0.1513, "caj_adp": 0.2188, "caj_acc": 0.9763, "agw_acc": 0.9999, "lr": 0.01, "fix_adp": 0.0134, "fix_acc": 0.9963}, {"epoch": 81, "caj_id": 0.1214, "caj_adp": 0.1686, "caj_acc": 0.9838, "agw_acc": 0.9999, "lr": 0.001, "fix_adp": 0.0102, "fix_acc": 0.9973}, {"epoch": 100, "caj_id": 0.1204, "caj_adp": 0.1583, "caj_acc": 0.984, "agw_acc": null, "lr": 0.001, "fix_adp": 0.0098, "fix_acc": 0.997}]')
show_table(["Epoch", "LR", "ADP loss (authors')", "Train ID acc. (authors')", "ADP loss (corrected)", "Train ID acc. (corrected)", "AGW train ID acc."], [
    [row["epoch"], f"{row['lr']:g}",
     f"{row['caj_adp']:.3f}", f"{row['caj_acc']:.3f}", f"{row['fix_adp']:.3f}", f"{row['fix_acc']:.3f}",
     "—" if row["agw_acc"] is None else f"{row['agw_acc']:.3f}"]
    for row in caj_curve])

| Epoch | LR | ADP loss (authors') | Train ID acc. (authors') | ADP loss (corrected) | Train ID acc. (corrected) | AGW train ID acc. |
| --- | --- | --- | --- | --- | --- | --- |
| 1 | 0.01 | 88.000 | 0.374 | 1.650 | 0.098 | 0.487 |
| 5 | 0.05 | 88.000 | 0.899 | 0.472 | 0.785 | 0.976 |
| 6 | 0.06 | 81.483 | 0.846 | 0.386 | 0.843 | 0.985 |
| 7 | 0.07 | 0.702 | 0.028 | 0.315 | 0.875 | 0.993 |
| 10 | 0.1 | 0.689 | 0.213 | 0.210 | 0.909 | 0.992 |
| 20 | 0.1 | 0.472 | 0.803 | 0.063 | 0.969 | 0.879 |
| 50 | 0.01 | 0.219 | 0.976 | 0.013 | 0.996 | 1.000 |
| 81 | 0.001 | 0.169 | 0.984 | 0.010 | 0.997 | 1.000 |
| 100 | 0.001 | 0.158 | 0.984 | 0.010 | 0.997 | — |

For the first five epochs the ADP loss sat at its clamp value of 88, where its gradient is zero, so only the identity loss was learning. During epoch 6 the ADP term left the clamp and produced large gradients. By epoch 7, training identity accuracy had fallen from 0.85 to 0.03, close to chance (1/296). ADP settled near ln 2 ≈ 0.69, meaning the positive and negative distances had become equal: the features had collapsed. The model relearned the training identities (98% accuracy by epoch 100) but reached only 7.0% Rank-1 on validation, below its own 2-epoch pilot (11.8%).

The cause lies in the hard-sample weighting that AGW and CAJ share. When computing softmax weights over negatives, the code takes the maximum over a row that still contains masked-out zeros. With pre-BN features of norm about 50, every negative weight underflows, so the closest-negative distance evaluates to about 0. ADP then only pulls positives together, and nothing pushes different identities apart. AGW uses the same weighting, but its WRT loss has bounded gradients, and it trained normally. Taking the maximum over valid entries only restores real negative distances (about 26 at initialisation, instead of 0).

We retrained CAJ with only this change (`masked_max` in `src/agw.py`, configuration `CAJ_fix`); data, augmentation, losses, schedule and seed are unchanged. A 10-epoch pilot passed the epoch-6 point where the original run collapsed and reached 21.1% Rank-1, so the run was continued to 100 epochs. The ADP loss now starts at 1.65 rather than the clamp value and decreases steadily, training identity accuracy rises without a drop, and no loss value was non-finite. The final model reaches 34.4% mAP and 33.1% Rank-1. Because the code was changed, this row is our corrected implementation of CAJ, not a reproduction of the authors' code.

## 6. Discussion

The baseline learns its training objectives and provides a common starting point for the proposed extensions. Its two-stage design first learns from grey RGB/IR inputs and then introduces colour and cross-modality losses.

On the internal validation split, full PMT (M0) and the corrected CAJ are the two strongest baselines and cannot be separated with one seed: their mAP is equal and the gallery-level differences change sign. M0 leads the plain ViT (V0) on every gallery draw, which points to a real but small contribution from PMT's components. It also leads the CNN baseline AGW by about 4 mAP points, at roughly 3.6 times the computation. The corrected CAJ reaches M0's accuracy with a ResNet-50, which makes it the stronger reference for the extensions: the smooth-transition extension also targets colour variation, and must improve on M0 by more than the gallery spread to show a benefit beyond what channel augmentation already gives a CNN. All results come from one training seed, so repeat seeds are needed before treating these margins as reliable.

The CAJ run shows that published code can be numerically fragile outside its original setting. A masking detail that is harmless for AGW's loss made CAJ's loss collapse the features, and only the training curve and a pilot evaluation exposed it. Correcting that one detail raised CAJ from 7.0% to 33.1% Rank-1. We therefore evaluate a short pilot before every full run and inspect each loss term rather than the total alone.

Training identities, camera coverage (cam3 queries against cam4/5 galleries only) and test-time settings differ from the published setting, so published benchmark scores should not be compared directly with these numbers.

Next, we will complete the controlled extension comparisons (M1–M3) with the same evaluator, confirm with the teaching staff that the corrected CAJ is acceptable as a baseline, and add difficult-query examples.

### References

[1] Lu, H., Zou, X., & Zhang, P. (2023). *Learning Progressive Modality-shared Transformers for Effective Visible-Infrared Person Re-identification*. AAAI. [Paper](https://arxiv.org/abs/2212.00226); [author code, pinned version](https://github.com/hulu88/PMT/tree/0b31da97b06428473bc4a1134b322ea226340818).

[2] [SYSU-MM01 dataset and evaluation protocol](https://github.com/wuancong/SYSU-MM01); [course-linked data mirror](https://www.kaggle.com/datasets/coconutjean/sysumm01).

[3] Ye, M., Shen, J., Lin, G., Xiang, T., Shao, L., & Hoi, S. C. H. (2021). *Deep Learning for Person Re-identification: A Survey and Outlook*. IEEE TPAMI. [Author code (AGW), pinned version](https://github.com/mangye16/Cross-Modal-Re-ID-baseline/tree/e817a2672c1e14e8802617acb980a79520ef9545).

[4] Ye, M., Ruan, W., Du, B., & Shou, M. Z. (2021). *Channel Augmented Joint Learning for Visible-Infrared Recognition*. ICCV. [Author code (CAJ), same repository, `ICCV21_CAJ`](https://github.com/mangye16/Cross-Modal-Re-ID-baseline/tree/e817a2672c1e14e8802617acb980a79520ef9545/ICCV21_CAJ).
